In [1]:
#using word2Vec
import numpy as np
import pandas as pd

In [2]:
temp_df = pd.read_csv('IMDB Dataset.csv')

In [3]:
#we are taking or giving only 20k out of 50k to model because code slow run hoga
#for improving accuracy give more data to model for training
df = temp_df.iloc[:20000]

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df['sentiment'].value_counts()

sentiment
negative    10097
positive     9903
Name: count, dtype: int64

In [6]:
#koi null value h ya nhi? nhi h
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [7]:
#there are 74 duplicate reviews in our data, so delete krdo
df.duplicated().sum()

np.int64(74)

In [8]:
#delete duplicate reviews
df.drop_duplicates(inplace= True)

/var/folders/g1/_0z_qznx7zs0xrj0v7h5x4sr0000gn/T/ipykernel_7299/2420708789.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace= True)


In [9]:
#again check? duplicate
df.duplicated().sum()

np.int64(0)

In [10]:
#now basic text preprocessing
#--Remove tags
#--lowercase
#--remove stopwords
import re 
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/arshsharma/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [12]:
df['review'] = df['review'].apply(remove_tags)

/var/folders/g1/_0z_qznx7zs0xrj0v7h5x4sr0000gn/T/ipykernel_7299/2336150696.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(remove_tags)


In [13]:
df['review'] = df['review'].apply(lambda x:x.lower())

/var/folders/g1/_0z_qznx7zs0xrj0v7h5x4sr0000gn/T/ipykernel_7299/740760900.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(lambda x:x.lower())


In [15]:
from nltk.corpus import stopwords
sw_list = stopwords.words('english')
df['review']= df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

/var/folders/g1/_0z_qznx7zs0xrj0v7h5x4sr0000gn/T/ipykernel_7299/2341230865.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review']= df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))


In [16]:
df

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production. filming technique...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically there's family little boy (jake) thi...,negative
4,"petter mattei's ""love time money"" visually stu...",positive
...,...,...
19995,"ok. starters, taxi driver amazing. this, taxi ...",negative
19996,"sort hard say it, greatly enjoyed ""targets"" ""p...",negative
19997,still liked though. warren beatty fair comic b...,positive
19998,could still use black adder even today. imagin...,positive


In [17]:
import gensim

In [18]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess


In [19]:
story = []
for sent in df['review']:
    raw_sent = sent_tokenize(sent)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))
    

In [21]:
#now creating word2vec model
!pip install gensim
import gensim

  Using cached gensim-4.4.0-cp310-cp310-macosx_11_0_arm64.whl.metadata (8.5 kB)
  Using cached smart_open-8.0.1-py3-none-any.whl.metadata (24 kB)
Using cached gensim-4.4.0-cp310-cp310-macosx_11_0_arm64.whl (24.4 MB)
Using cached smart_open-8.0.1-py3-none-any.whl (73 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gensim]━━━━ 1/2 [gensim]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [23]:
model = gensim.models.Word2Vec(
    window = 10,
    min_count = 2
    )

In [24]:
model.build_vocab(story)

In [26]:
model.train(story, total_examples = model.corpus_count, epochs = model.epochs)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


(11756725, 12360490)

In [27]:
len(model.wv.index_to_key)

42888

In [28]:
#average word 2 vec
def document_vector(doc):
    #remove out of vocabulary words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis = 0)

In [29]:
document_vector(df['review'].values[0])

array([-0.00564938,  0.3432055 , -0.01095689,  0.05185078,  0.2722342 ,
       -0.62143004,  0.17498228,  0.6941209 ,  0.0160729 , -0.10733122,
       -0.11037717, -0.22944863,  0.00921246,  0.16954939, -0.03896829,
       -0.1827669 , -0.11157781, -0.3563957 ,  0.07644638, -0.34239808,
        0.17120937,  0.1230448 ,  0.05289228, -0.09090219, -0.18511276,
       -0.27899793, -0.02895683, -0.22573353, -0.16951728, -0.01185326,
        0.5053361 , -0.03133823, -0.07889391, -0.21150978, -0.05876806,
        0.34592962, -0.16984622, -0.18087175, -0.20163383, -0.62063265,
        0.18021351, -0.04864459,  0.00443553, -0.14119908,  0.22402333,
       -0.01918633, -0.3349488 , -0.11214948,  0.308307  ,  0.18553154,
        0.5020569 , -0.44538406, -0.03853934, -0.34711117, -0.26436684,
        0.32324952,  0.56664526,  0.23791961, -0.31464177, -0.06205069,
        0.02480969,  0.24765132, -0.05706567,  0.10512777, -0.49504083,
        0.31071135, -0.04751039,  0.10693166, -0.6770292 ,  0.29

In [30]:
from tqdm import tqdm

In [31]:
X = []
for doc in tqdm(df['review'].values):
    X.append(document_vector(doc))

100%|█████████████████████████████████████████████████████████████████████████| 19926/19926 [09:16<00:00, 35.82it/s]


In [32]:
X = np.array(X)

In [33]:
X.shape

(19926, 100)

In [34]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

y = encoder.fit_transform(df['sentiment'])


In [35]:
y

array([1, 1, 1, ..., 1, 1, 0], shape=(19926,))

In [36]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 1)

In [37]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

In [38]:
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [40]:
y_pred = rf.predict(X_test)

In [41]:
from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(y_test, y_pred) 

0.8060712493728048